# TR-PEIS procedure

Notebook for measuring time-resolved potentiodynamic EIS (TR-PEIS) on Bio-Logic
potentiostats. Individual isofrequency PEIS responses are recorded during
repeated, identically initiated transient events (OCV hold → voltage step → PEIS
at a single frequency); interpolating the responses reconstructs instantaneous
impedance spectra with sub-second temporal resolution.

Method: *Beyond Steady-State Limitations of Electrochemical Impedance Spectroscopy:
Time-Resolved Decoupling of Transient Processes* — preprint 10.5281/zenodo.22961107,
dataset 10.5281/zenodo.22961027.

**How to use:** run the cells top to bottom on Windows with the EC-Lab Development
Package installed (set `ECLIB_DIR` to its `lib` folder). First install the
environment with `uv sync`, then start Jupyter with
`uv run jupyter notebook TR-PEIS_procedure.ipynb`. See README.md for details:
1. Connect to the potentiostat (set `address` and `channel`),
2. Adjust the experiment parameters (voltage, timing, frequency sweep, stabilization),
3. Set the output folder and file names,
4. Run the stabilization loops, then the TR-PEIS measurement cells.
Review all parameters before running the measurement cells — they start the
instrument immediately.

In [1]:
import time
from IPython.display import display
import kbio.kbio_run_techniques as techs
import kbio.kbio_run
from kbio.kbio_tech import get_info_data
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

Connect to the potentiostat and prepare the instrument for running techniques.

The `connect_potentiostat()` helper from `kbio.kbio_run`:
- loads the EcLab DLL from `ECLIB_DIR` or the default path
- connects to the potentiostat at the specified `address` and `channel`
- detects the board type and prints the recommended `.ecc` technique file set
- optionally loads firmware when `load_firmware=True`
- checks that the channel kernel is loaded before returning the API object

In [2]:
address = "192.168.0.61"
channel = 1
# Connect to the potentiostat. This function loads the EcLab DLL, connects to the
# specified instrument, detects the board type, prints which .ecc files should be
# used for the selected technique family, and optionally loads firmware.
api, id_ = kbio.kbio_run.connect_potentiostat(address, channel, load_firmware=True, path = r'\\NOTEB-NTB1-PEM\Science\Automation\EC-Lab Development Package\lib')
board_type = api.GetChannelBoardType(id_, channel)

> EcLib version: 6.11.2.0

> device[192.168.0.61] info :
SP150E 32MB, CPU=9200, 1 channel, 1 slot
Firmware: v28.00 2025/5/14
1 connection, HTdisplay on

> Board type detected: ESSENTIAL
Use technique.ecc files
> Loading kernel.bin ...
> ... firmware loaded

> Channel 1 info :
Channel: 1
C437_VMP3EZ board, S/N 81271
no LC head
with EIS capabilities
0 technique
State: STOP
no amplifiers
IRange: [I_RANGE_10uA, I_RANGE_BOOSTER]
MaxBandwidth: 4
Memory: 768.0KB (0.00% filled)
KERNEL (v5.291), FPGA (A611)

Potentiostat 192.168.0.61 successfully connected! You can load procedures!


In [7]:
# EIS measurement setup parameters
# - voltage: target cell voltage for the experiment
# - voltage_time: hold time at the set voltage for both CA and PEIS
# - ocv_time: open-circuit hold time used between CA and PEIS
voltage = 1.46
voltage_time = 30
ocv_time = 60

# Frequency sweep settings for PEIS
points_per_decade = 6
f_start = 20000
f_end = 2

# Advanced frequency list options for state validation:
# - repeat_percentage: Percentage of frequencies to repeat (0-100) for drift detection
# - checkpoint_freq: Frequency to insert at regular intervals as a checkpoint
# - checkpoint_interval: Insert checkpoint every N frequencies
# - random_seed: Seed for reproducibility (None for random)
repeat_percentage = 20.0  # Repeat 20% of frequencies
checkpoint_freq = 1000.0   # Checkpoint frequency in Hz (None to disable)
checkpoint_interval = 5    # Insert checkpoint every N frequencies (None to disable)
random_seed = None        # None for random, or integer for reproducibility

# Generate frequency list with randomization, repeats, and checkpoints
f_list, freq_info = techs.create_frequency_list(
    f_start=f_start,
    f_end=f_end,
    points_per_decade=points_per_decade,
    repeat_percentage=repeat_percentage,
    checkpoint_freq=checkpoint_freq,
    checkpoint_interval=checkpoint_interval,
    random_seed=random_seed
)

# Print summary of frequency list
print(f'Generated {freq_info["total_measurements"]} measurements:')
print(f'  Base frequencies: {freq_info["num_base"]}')
if freq_info["num_repeats"] > 0:
    print(f'  Repeated frequencies: {freq_info["num_repeats"]} ({freq_info["repeat_percentage"]}%)')
    print(f'  Frequencies to repeat: {freq_info["frequencies_to_repeat"]}')
if freq_info["checkpoint_frequency"] is not None:
    print(f'  Checkpoint frequency: {freq_info["checkpoint_frequency"]} Hz, every {freq_info["checkpoint_interval"]} freqs')
print(f'Frequency list: {f_list}')

# Pre-PEIS state checkpoint settings
# Run a quick PEIS measurement at a specific frequency before each main measurement
# to verify the electrochemical state. This can be enabled for:
# - Initial stabilization CA cycles (state_check_before_init_ca)
# - Main PEIS measurements (state_check_before_peis)
state_check_freq = 1000.0    # Frequency for state checkpoint in Hz (None to disable)
state_check_repeats = 2      # Number of repetitions for state checkpoint
state_check_before_init_ca = False  # Run state checkpoint before CA in initialization
state_check_before_peis = False    # Run state checkpoint before PEIS in main measurement

# Maximum and minimum number of initial OCV and CA cycles before current is stabilized
repeats_max = 20
repeats_min = 5

# Set how much close two subsequent currents must be to each other to be considered as stable in percent.
# E.g. 0.2 means that 0.998 < average(I_(n+1)/I_n) < 1.002
stabilization_range = 0.1

# Create the technique parameter objects used by the instrument API
ca_params = techs.CA_params(
    api,
    Voltage_step=voltage,
    vs_init=False,
    Duration_step=voltage_time,
    Record_every_dT=0.1,
    Record_every_dI=1,
    N_cycles=0,
    I_range=11,
    E_range=3,
)
# OCV technique parameters define how long the open-circuit hold lasts
ocv_params = techs.OCV_params(api, Rest_time_T=ocv_time, Record_every_dT=0.1, Record_every_dE=1)

# LOOP parameters determine the protocol sequencing for the technique chain
loop_params = techs.LOOP_params(api, loop_N_times=-1, protocol_number=1)
loop_params_init = techs.LOOP_params(api, loop_N_times=-1, protocol_number=0)

# Helper function to run a state checkpoint PEIS measurement
def run_state_checkpoint(api, channel, id_, board_type, freq, repeats=1):
    """Run a quick PEIS measurement at a specific frequency for state verification."""
    checkpoint_peis_params = techs.PEIS_params(
        api,
        vs_init=False,
        Initial_Voltage_step=voltage,
        step_duration=0,
        record_dt=1,
        record_dE=1,
        Final_frequency=freq,
        Initial_frequency=freq,
        Lin_Log=True,
        Amplitude=0.005,
        Frequency_number=1,
        Average_N_times=repeats,
        Correction=False,
        Wait_for_steady=0.1,
        I_range=11,
    )
    
    # Load and run the checkpoint PEIS
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)
    api.LoadTechnique(id_, channel, 'peis.ecc', checkpoint_peis_params, first=False, last=True)
    api.StartChannel(id_, channel)
    
    # Wait for completion and collect data
    checkpoint_data = None
    while True:
        data = api.GetData(id_, channel)
        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)
            if status == 'STOP':
                checkpoint_data = data
                break
    api.StopChannel(id_, channel)
    return checkpoint_data

[2.00000000e+04 1.34003750e+04 8.97850252e+03 6.01576504e+03
 4.03067537e+03 2.70062808e+03 1.80947145e+03 1.21237980e+03
 8.12317198e+02 5.44267754e+02 3.64669600e+02 2.44335470e+02
 1.63709346e+02 1.09688332e+02 7.34932388e+01 4.92418480e+01
 3.29929615e+01 2.21059028e+01 1.48113694e+01 9.92389521e+00
 6.64919586e+00 4.45508590e+00 2.98499109e+00 2.00000000e+00]


### Control Experiment: Aging Characterization

Run this cell to determine the maximum number of transient cycles that can be
performed before the cell shows detectable degradation. This helps establish
a safe limit for the main TR-PEIS experiment.

The experiment runs repeated OCV-CA cycles and measures:
- Current response stability
- Reference impedance at a specified frequency (optional)

The test stops when:
- The current or impedance change exceeds the specified threshold, OR
- The maximum number of cycles is reached

**Output:** Number of safe cycles before degradation is detectable.

In [ ]:
# Aging characterization experiment parameters
aging_max_cycles = 50          # Maximum number of cycles to test (safety limit)
aging_current_threshold = 0.01  # Stop if current changes by more than 1%
aging_z_threshold = 0.02        # Stop if impedance changes by more than 2%
aging_ref_freq = 1000.0         # Frequency for reference impedance measurement (Hz)
aging_measure_z = True          # Set to False for faster test (current only)

# Helper function to run a single aging cycle
def run_aging_cycle(cycle_num):
    """Run one OCV-CA cycle and optionally measure reference impedance."""
    # Run OCV -> CA
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)
    api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=True)
    api.StartChannel(id_, channel)
    
    # Collect CA data
    ca_data = None
    while True:
        data = api.GetData(id_, channel)
        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)
            if status == 'STOP':
                ca_data = data
                break
    api.StopChannel(id_, channel)
    
    # Optionally measure reference impedance
    z_data = None
    if aging_measure_z and aging_ref_freq is not None:
        z_data = run_state_checkpoint(api, channel, id_, board_type, aging_ref_freq, 1)
    
    return ca_data, z_data

# Run aging characterization
print(f'Starting aging characterization test (max {aging_max_cycles} cycles)...')
print(f'Current threshold: {aging_current_threshold*100:.1f}%, Impedance threshold: {aging_z_threshold*100:.1f}%')

# Storage for results
current_values = []  # Average current from each CA cycle
impedance_values = []  # Impedance at reference frequency from each cycle
cycle_times = []  # Elapsed time for each cycle

# Run first cycle to establish baseline
ca_data, z_data = run_aging_cycle(0)

# Extract current from CA data
from kbio.kbio_run import get_exp_data
temp_data_out = kbio.kbio_run.create_data_out(['CA'])
temp_data_out = get_exp_data(api, ca_data, board_type=board_type, data_out=temp_data_out, index=0)
baseline_current = np.mean(np.array(temp_data_out['CA']['I/mA'])[np.array(temp_data_out['CA']['time/s']) > 1])
current_values.append(baseline_current)

# Extract impedance if measured
if z_data is not None:
    temp_data_out_z = kbio.kbio_run.create_data_out(['PEIS'])
    temp_data_out_z = get_exp_data(api, z_data, board_type=board_type, data_out=temp_data_out_z, index=0)
    baseline_impedance = np.mean(np.array(temp_data_out_z['PEIS']['Z/Ohm']))
    impedance_values.append(baseline_impedance)
else:
    baseline_impedance = None

cycle_times.append(0)
print(f'Cycle 0: Current = {baseline_current:.4f} mA, Impedance = {baseline_impedance:.2f} Ω' if baseline_impedance else f'Cycle 0: Current = {baseline_current:.4f} mA')

# Run remaining cycles
for cycle in range(1, aging_max_cycles):
    ca_data, z_data = run_aging_cycle(cycle)
    
    # Extract current
    temp_data_out = kbio.kbio_run.create_data_out(['CA'])
    temp_data_out = get_exp_data(api, ca_data, board_type=board_type, data_out=temp_data_out, index=cycle)
    current = np.mean(np.array(temp_data_out['CA']['I/mA'])[np.array(temp_data_out['CA']['time/s']) > 1])
    current_values.append(current)
    
    # Extract impedance if measured
    if z_data is not None:
        temp_data_out_z = kbio.kbio_run.create_data_out(['PEIS'])
        temp_data_out_z = get_exp_data(api, z_data, board_type=board_type, data_out=temp_data_out_z, index=cycle)
        impedance = np.mean(np.array(temp_data_out_z['PEIS']['Z/Ohm']))
        impedance_values.append(impedance)
    
    cycle_times.append(cycle)
    
    # Calculate changes from baseline
    current_change = abs(current - baseline_current) / abs(baseline_current)
    z_change = None
    if baseline_impedance is not None:
        z_change = abs(impedance - baseline_impedance) / abs(baseline_impedance)
    
    # Print progress
    if z_change is not None:
        print(f'Cycle {cycle}: Current = {current:.4f} mA (Δ={current_change*100:.2f}%), Impedance = {impedance:.2f} Ω (Δ={z_change*100:.2f}%)')
    else:
        print(f'Cycle {cycle}: Current = {current:.4f} mA (Δ={current_change*100:.2f}%)')
    
    # Check thresholds
    if current_change > aging_current_threshold:
        print(f'\n⚠️  STOPPING: Current changed by {current_change*100:.2f}% > {aging_current_threshold*100:.1f}% threshold')
        break
    if z_change is not None and z_change > aging_z_threshold:
        print(f'\n⚠️  STOPPING: Impedance changed by {z_change*100:.2f}% > {aging_z_threshold*100:.1f}% threshold')
        break

# Results
safe_cycles = len(current_values) - 1  # Subtract 1 because we start from cycle 0
print(f'\n{'='*60}')
print(f'Aging Characterization Results')
print(f'{'='*60}')
print(f'Maximum safe cycles before degradation: {safe_cycles}')
print(f'Final current change: {current_change*100:.2f}%')
if baseline_impedance is not None:
    print(f'Final impedance change: {z_change*100:.2f}%')
print(f'{'='*60}')

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Current plot
ax1.plot(cycle_times, current_values, 'o-', label='Current')
ax1.axhline(y=baseline_current * (1 + aging_current_threshold), color='r', linestyle='--', label=f'{aging_current_threshold*100:.1f}% threshold')
ax1.axhline(y=baseline_current * (1 - aging_current_threshold), color='r', linestyle='--')
ax1.set_xlabel('Cycle')
ax1.set_ylabel('Current (mA)')
ax1.set_title('Current vs Cycle')
ax1.legend()
ax1.grid(True)

# Impedance plot (if measured)
if impedance_values:
    ax2.plot(cycle_times, impedance_values, 'o-', label='Impedance', color='green')
    ax2.axhline(y=baseline_impedance * (1 + aging_z_threshold), color='r', linestyle='--', label=f'{aging_z_threshold*100:.1f}% threshold')
    ax2.axhline(y=baseline_impedance * (1 - aging_z_threshold), color='r', linestyle='--')
    ax2.set_xlabel('Cycle')
    ax2.set_ylabel('Impedance (Ω)')
    ax2.set_title('Impedance vs Cycle')
    ax2.legend()
    ax2.grid(True)
else:
    ax2.axis('off')
    ax2.text(0.5, 0.5, 'Impedance measurement\ndisabled', ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.show()

# Return the safe number of cycles
safe_cycles

In [8]:
voltage_str = str(voltage).replace('.', ',')  # convert voltage to string and replace decimal point with comma
ocv_time_str = str(ocv_time).replace('.', ',')  # convert ocv_time to string and replace decimal point with comma
voltage_time_str = str(voltage_time).replace('.', ',')  # convert voltage_time to string and replace decimal point with comma

# Path for saving output files
output_folder = r'C:\PEM-WE_measurements\2026\480_VIII_cathode_etching_series_210min_more_C\trpeis test'  
output_file = "trpeis_at_" + voltage_str + "V_for_" + voltage_time_str + "s_and_" + ocv_time_str + "s_OCV_"  # build base filename with experiment parameters
output_file_init = output_file + "_init"  # create a filename for data from initial stabilization cycles

  # list existing files that match the output file base name, so you can adjust the filename so new files are created for the new measurement
kbio.kbio_run.print_matching_files(output_folder, output_file)


Files in C:\PEM-WE_measurements\2026\480_VIII_cathode_etching_series_210min_more_C\trpeis test containing 'trpeis_at_1,46V_for_30s_and_60s_OCV_':
trpeis_at_1,46V_for_30s_and_60s_OCV__initCA.txt
trpeis_at_1,46V_for_30s_and_60s_OCV__initOCV.txt


In [ ]:
"""
Initial loops for system stabilization
"""
plot_output = widgets.Output()
display(plot_output)

# Choose initialization method based on state checkpoint settings
if state_check_before_init_ca and state_check_freq is not None:
    # Modified initialization with state checkpoint before each CA
    techniqs = ['CA','OCV']
    index = 0
    data_out_init_all = []
    
    while index < repeats_max:
        # Run state checkpoint before CA
        print(f'\nRunning state checkpoint before CA cycle {index + 1}...')
        checkpoint_data = run_state_checkpoint(api, channel, id_, board_type, state_check_freq, state_check_repeats)
        print(f'State checkpoint completed.')
        
        # Run OCV -> CA sequence
        api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)
        api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=True)
        api.StartChannel(id_, channel)
        
        data_out_init = kbio.kbio_run.create_data_out(techniqs)
        while True:
            data = api.GetData(id_, channel)
            if len(data[2]) > 0:
                status, tech_name = get_info_data(api, data)
                data_out_init = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out_init, index=index)
                print('\rInitial loops running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end='', flush=True)
                time.sleep(1)
                if status == 'STOP':
                    break
        api.StopChannel(id_, channel)
        
        # Save data and check stability
        data_out_init_all.append(data_out_init)
        save_data = kbio.kbio_run.save_data(data_out_init, output_folder, output_file_init)
        
        if index >= repeats_min - 1:
            mask1 = np.array(data_out_init_all[index]['CA']['time/s']) > 1
            mask2 = np.array(data_out_init_all[index-1]['CA']['time/s']) > 1
            avg = np.mean(np.array(data_out_init_all[index]['CA']['I/mA'])[mask1] / np.array(data_out_init_all[index-1]['CA']['I/mA'])[mask2])
            print(avg)
            if 1-stabilization_range/100 < avg < 1+stabilization_range/100:
                print(f'Current stabilized in cycle {index}')
                break
        
        index += 1
        
        # Update plot
        with plot_output:
            plot_output.clear_output(wait=True)
            fig = kbio.kbio_run.plot_results(data_out_init, techniqs, index)
            display(fig)
            plt.close(fig)
        
        data_out_init = kbio.kbio_run.create_data_out(techniqs)
    
    print('\nInitial loops with state checkpoints finished.')
else:
    # Original initialization without state checkpoints
    techniqs = ['CA','OCV']  # techniques to include for the initial stabilization phase

    index = 0  # loop index for the current data segment
    data_out_init = kbio.kbio_run.create_data_out(techniqs)  # initialize the data structure for initial loops
    data_out_init_all = []  # store all initial loop data sets for stability checking

    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # load OCV technique first
    api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=False)  # load CA technique next
    api.LoadTechnique(id_, channel, 'loop.ecc', loop_params_init, first=False, last=True)  # load loop controller for the init sequence

    api.StartChannel(id_, channel)  # start the instrument channel for initial loops

    while True:
        data = api.GetData(id_, channel)  # fetch the latest data package from the instrument

        if len(data[2]) > 0:  # proceed only when there is actual recorded data
            status, tech_name = get_info_data(api, data)  # decode technique and status
            data_out_init = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out_init, index=index)  # parse and append data

            print('\rInitial loops running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end='', flush=True)  # live status update

            time.sleep(1)  # throttle updates during the initial loops

            if index < data[1].loop:  # only save loop data when the loop index is valid
                data_out_init_all.append(data_out_init)  # keep a copy of the current initial loop data

                if index > repeats_min - 1:  # At least three stabilization cycles has to run
                    mask1 = np.array(data_out_init_all[index]['CA']['time/s']) > 1  # ignore first second of CA data
                    mask2 = np.array(data_out_init_all[index-1]['CA']['time/s']) > 1  # ignore first second of previous CA cycle

                    # compare current stabilization between consecutive CA cycles
                    avg = np.mean(np.array(data_out_init_all[index]['CA']['I/mA'])[mask1] / np.array(data_out_init_all[index-1]['CA']['I/mA'])[mask2])
                    print(avg)
                    if 1-stabilization_range/100 < avg < 1+stabilization_range/100:
                        print(f'Current stabilized in cycle {index}')
                        break

                index = data[1].loop  # update index from the instrument loop counter

                with plot_output:
                    plot_output.clear_output(wait=True)  # clear previous plot output
                    fig = kbio.kbio_run.plot_results(data_out_init, techniqs, index)  # generate live plot
                    display(fig)
                    plt.close(fig)

                save_data = kbio.kbio_run.save_data(data_out_init, output_folder, output_file_init)  # save current init data to disk
                data_out_init = kbio.kbio_run.create_data_out(techniqs)  # reset the init data buffer for the next loop

            if index > repeats_max - 1:  # safety break to avoid indefinite initial looping
                break

    api.StopChannel(id_, channel)    
    print('Initial loops finished. ')
"""
Impedance spectroscopy measurement at different frequencies
"""
techniqs = ['OCV','CA','PEIS']
# list to accumulate results from each frequency
data_out_all = []

for index, f in enumerate(f_list):
    data_out = kbio.kbio_run.create_data_out(techniqs)  # create an empty data container for this frequency


    # Load pre-conditioning techniques: OCV then CA
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # hold voltage before conditioning
    api.LoadTechnique(id_, channel, 'ca.ecc', ca_params, first=False, last=True)  # current application for stabilization

    api.StartChannel(id_, channel)  # start the channel to execute the queued techniques

    print(f"Reading data for {f} Hz")  # announce the frequency being measured

    while True:
        data = api.GetData(id_, channel)  # poll the instrument for new data

        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)  # decode technique name and status from the packet
            data_out = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out, index=index)  # parse and append data

            # live progress output; carriage return keeps the print on one line
            print('\rFrequency ' + str(f) + ' Hz running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end="", flush=True)
            if tech_name == 'OCV':
                time.sleep(1)  # pause briefly during OCV updates to reduce CPU load
            elif tech_name == 'CA':
                time.sleep(1)  # pause briefly during CA updates

            if status == "STOP":
                break  # pre-conditioning finished, exit loop
    api.StopChannel(id_, channel)  # stop the channel after pre-conditioning

    # Run state checkpoint before PEIS if enabled
    if state_check_before_peis and state_check_freq is not None:
        print(f'  Running state checkpoint at {state_check_freq} Hz...')
        checkpoint_data = run_state_checkpoint(api, channel, id_, board_type, state_check_freq, state_check_repeats)
        print(f'  State checkpoint completed.')

    # Prepare and run PEIS at the current frequency
    api.LoadTechnique(id_, channel, 'ocv.ecc', ocv_params, first=True, last=False)  # ensure voltage hold before PEIS
    peis_params = techs.PEIS_params(
        api,
        vs_init=False,
        Initial_Voltage_step=voltage,
        step_duration=0,
        record_dt=1,
        record_dE=1,
        Final_frequency=f,
        Initial_frequency=f,
        Lin_Log=True,
        Amplitude=0.005,
        Frequency_number=1,
        Average_N_times=1,
        Correction=False,
        Wait_for_steady=0.1,
        I_range=11,
    )  # configure PEIS parameters specific to this frequency
    api.LoadTechnique(id_, channel, 'peis.ecc', peis_params, first=False, last=False)  # load PEIS technique
    api.LoadTechnique(id_, channel, 'loop.ecc', loop_params, first=False, last=True)  # load loop controller to finish sequence

    # Start PEIS sequence
    api.StartChannel(id_, channel)

    while True:
        data = api.GetData(id_, channel)  # read data while PEIS runs

        if len(data[2]) > 0:
            status, tech_name = get_info_data(api, data)  # get current technique/status
            # extend elapsed time to account for hold durations
            if data[0].ElapsedTime < (voltage_time + ocv_time):
                data[0].ElapsedTime += (voltage_time + ocv_time)  # adjust elapsed time for display
                data_out = kbio.kbio_run.get_exp_data(api, data, board_type=board_type, data_out=data_out, index=index)  # append PEIS data

                # live PEIS progress update
                print('\rFrequency ' + str(f) + ' Hz running at ' + str(tech_name) + ' for ' + str(round(data[0].ElapsedTime, 4)) + ' s', end="", flush=True)
                if tech_name == 'OCV':
                    time.sleep(1)  # throttle updates during OCV
                else:
                    pass  # no delay for other states
            else:
                print(" ")
                print(f'Measurement at {f} Hz finished.')  # end-of-frequency message
                break
    api.StopChannel(id_, channel)  # stop channel when PEIS completes

    data_out_all.append(data_out)  # collect this frequency's data

    with plot_output:
        plot_output.clear_output(wait=True)  # clear prior plot
        fig = kbio.kbio_run.plot_results(data_out_all, techniqs, index)  # plot accumulated results so far
        display(fig)
        plt.close(fig)

    save_data = kbio.kbio_run.save_data(data_out, output_folder, output_file)  # save the current data files
    index += 1  # increment measurement counter
api.Disconnect(id_)  # disconnect from instrument at end of sweep


print("Experiment completed")



Output()

Initial loops running at OCV for 91.511 s

Initial loops running at OCV for 182.171 s

Initial loops running at OCV for 271.847 s

Initial loops running at OCV for 361.56 ss

Initial loops running at OCV for 452.206 s

Initial loops running at OCV for 541.876 s1.0021330303117968


Initial loops running at OCV for 632.55 ss1.0248630990440535


Initial loops running at OCV for 722.26 ss0.9778980649526704


Initial loops running at OCV for 811.928 s1.0135457253644944


Initial loops running at CA for 882.467 ss